In [ ]:
import sys
# from pathlib import Path
# Ensure src/ is in path (same convention as notebooks)
# sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

sys.path.insert(0, '../src')

import numpy as np
from models.base import HamiltonianModel
from models.graphene import (
    SingleLayerGrapheneTB,
    BilayerGrapheneTB,
    SingleLayerGrapheneKP,
    BilayerGrapheneKP,
    BilayerGrapheneKPAA,
)


In [ ]:
"""
验证 K 点附近光学矩阵元 v^alpha_{cv}(k) = <c,k| v_alpha |v,k>。

检测项：
1. 各向同性：|v_{cv}| = vF，与绕 K 的角度无关
2. 角度依赖：v^x_{cv} ~ sin(phi), v^y_{cv} ~ cos(phi)
3. 连续性：穿过 K 点时光滑无跳变
4. 时间反演：K 和 K' 谷的矩阵元满足 v(K') = -v(K)^*
"""
results = []
vF = np.sqrt(3) / 2 * 2.78  # ~2.4076
tol_iso = 0.02   # 各向同性容忍度 (TB 有微小 trigonal warping)
tol_kp = 0.001   # KP 模型应严格

# ── 1. KP 模型 — 严格各向同性 ───────────────────────
kp_K = SingleLayerGrapheneKP(t=2.78, valley=+1)
K_pt = kp_K.K_point
dk = 0.01
n_angles = 24
velocities_kp = []
phases_kp = []

for i in range(n_angles):
    phi = 2 * np.pi * i / n_angles
    k = K_pt + dk * np.array([np.cos(phi), np.sin(phi)])
    E, V = kp_K.solve(k)
    vx, vy = kp_K.velocity_operator(k)
    # v^{alpha}_{cv} = V[:,0]^dagger @ v_alpha @ V[:,1]  (0=valence, 1=conduction)
    vx_cv = V[:, 0].conj() @ vx @ V[:, 1]
    vy_cv = V[:, 0].conj() @ vy @ V[:, 1]
    v_abs = np.sqrt(np.abs(vx_cv)**2 + np.abs(vy_cv)**2)
    velocities_kp.append(v_abs)
    phases_kp.append((phi, vx_cv, vy_cv))

v_std = np.std(velocities_kp) / np.mean(velocities_kp)
v_mean = np.mean(velocities_kp)
print(v_mean, vF)

results.append((
    f"KP isotropy: mean|v_cv|={v_mean:.4f} (vF={vF:.4f}), rel std={v_std:.2e}",
    np.allclose(v_mean, vF, atol=tol_kp) and v_std < 0.01
))


In [ ]:

# 角度依赖: vx_cv / sin(phi) 应为常数, vy_cv / cos(phi) 也应为常数
vx_norm = []
vy_norm = []
for phi, vx_cv, vy_cv in phases_kp:
    if abs(np.sin(phi)) > 0.1:
        vx_norm.append(abs(vx_cv / np.sin(phi)))
    if abs(np.cos(phi)) > 0.1:
        vy_norm.append(abs(vy_cv / np.cos(phi)))

print(np.mean(vx_norm),np.mean(vy_norm))
vx_std = np.std(vx_norm) / np.mean(vx_norm) if vx_norm else 0
vy_std = np.std(vy_norm) / np.mean(vy_norm) if vy_norm else 0
results.append((
    f"KP angular: |vx/sin| const (rel std={vx_std:.2e}), |vy/cos| const (rel std={vy_std:.2e})",
    vx_std < 0.01 and vy_std < 0.01
))


In [ ]:

# ── 2. TB 模型 — 近各向同性（微小 trigonal warping）─
tb_K = SingleLayerGrapheneTB(t=2.78)
velocities_tb = []

for i in range(n_angles):
    phi = 2 * np.pi * i / n_angles
    k = K_pt + dk * np.array([np.cos(phi), np.sin(phi)])
    E, V = tb_K.solve(k)
    vx, vy = tb_K.velocity_operator(k)
    vx_cv = V[:, 0].conj() @ vx @ V[:, 1]
    vy_cv = V[:, 0].conj() @ vy @ V[:, 1]
    velocities_tb.append(np.sqrt(np.abs(vx_cv)**2 + np.abs(vy_cv)**2))

v_tb_std = np.std(velocities_tb) / np.mean(velocities_tb)
v_tb_mean = np.mean(velocities_tb)
print(v_tb_mean)
results.append((
    f"TB isotropy: mean|v_cv|={v_tb_mean:.4f} (vF={vF:.4f}), rel std={v_tb_std:.2e}",
    np.allclose(v_tb_mean, vF, atol=tol_iso) and v_tb_std < 0.05
))


In [ ]:

# ── 3. 连续性：径向穿过 K 点 ─────────────────────────
n_radial = 100
ks = np.linspace(-0.05, 0.05, n_radial)
v_cv_along = []
for dq in ks:
    k = K_pt + np.array([dq, 0.0])
    E, V = tb_K.solve(k)
    vx, vy = tb_K.velocity_operator(k)
    vx_cv = V[:, 0].conj() @ vx @ V[:, 1]
    vy_cv = V[:, 0].conj() @ vy @ V[:, 1]
    v_cv_along.append((dq, np.abs(vx_cv), np.abs(vy_cv)))

# 连续性：相邻k点的矩阵元差分应小
vx_vals = np.array([v[1] for v in v_cv_along])
vy_vals = np.array([v[2] for v in v_cv_along])
print(vx_vals, vy_vals)
vx_jumps = np.max(np.abs(np.diff(vx_vals)))
vy_jumps = np.max(np.abs(np.diff(vy_vals)))
results.append((
    f"TB continuity: max jump |vx|={vx_jumps:.4f}, |vy|={vy_jumps:.4f}",
    vx_jumps < 0.1 and vy_jumps < 0.1
))


# Dirac 哈密顿量具有赝自旋-动量锁定（pseudospin-momentum locking）

- 本征态的赝自旋沿着 q。
- 带间跃迁矩阵元对应于改变赝自旋方向。
- 因此光学跃迁只耦合到垂直于 q 的偏振方向。

In [ ]:

# ── 4. 时间反演对称：K vs K' ─────────────────────────
kp_Kp = SingleLayerGrapheneKP(t=2.78, valley=-1)
Kp_pt = kp_Kp.Kp_point
# 在 K 谷取 k = K + dk, 在 K' 谷取 k = K' - dk (时间反演 partner)
dk_tr = np.array([0.01, 0.005])
k_test = K_pt + dk_tr
kp_test = Kp_pt - dk_tr

_, V_k = kp_K.solve(k_test)
vx_k, vy_k = kp_K.velocity_operator(k_test)
_, V_kp = kp_Kp.solve(kp_test)
vx_kp, vy_kp = kp_Kp.velocity_operator(kp_test)

# v^{alpha}_{cv}(K, k) = V_c^dag v_alpha V_v
vx_cv_K = V_k[:, 0].conj() @ vx_k @ V_k[:, 1]
vy_cv_K = V_k[:, 0].conj() @ vy_k @ V_k[:, 1]
vx_cv_Kp = V_kp[:, 0].conj() @ vx_kp @ V_kp[:, 1]
vy_cv_Kp = V_kp[:, 0].conj() @ vy_kp @ V_kp[:, 1]

print(vx_cv_K, vx_cv_Kp, vy_cv_K, vy_cv_Kp)

# T 对称: v(K') = -v(K)^* (时间反演下速度反号)
diff_tr_x = np.abs(vx_cv_Kp + np.conj(vx_cv_K))
diff_tr_y = np.abs(vy_cv_Kp + np.conj(vy_cv_K))
results.append((
    f"Time-reversal: |v(K')+v(K)*| = ({diff_tr_x:.2e}, {diff_tr_y:.2e})",
    diff_tr_x < 0.01 and diff_tr_y < 0.01
))


In [ ]:

# ── 输出 ─────────────────────────────────────────────
if True:
    print("\n" + "=" * 52)
    print("  Optical Matrix Element Verification")
    print("=" * 52)
    passed = 0
    for desc, ok in results:
        status = "PASS" if ok else "FAIL!"
        print(f"  [{status}] {desc}")
        if ok:
            passed += 1
    print("-" * 52)
    print(f"  Passed: {passed}/{len(results)}")
    print("=" * 52)
